# RiskSequencer — 04 Error Analysis

Where does the model fail, and what does it cost? (build-plan Phase 3.3–3.4)

1. Confusion matrix at the deployed threshold (FPR ≤ 5%)
2. **False negatives** — which fraud patterns slip through?
3. **False positives** — which legit behavior looks fraudulent?
4. **Business metrics** — $ fraud caught vs false-positive cost vs net savings
5. **Profit curve** — the threshold that maximises net savings (often ≠ the
   FPR-based threshold)

Runs on synthetic or the real IEEE-CIS parquet.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from config import FEATURE_COLUMNS, MONITORED_FEATURES, LABEL_COL, AMOUNT_COL, RAW_DIR, TrainConfig
from features.feature_pipeline import build_features
from data.sequence_builder import build_sequences, fit_scaler, time_based_split
from training.train import run_training, _scores
from training.evaluate import evaluate, tune_threshold
from training.business_metrics import business_metrics, net_savings_by_threshold

## 1. Load data, train, score the held-out test set

In [ ]:
parquet = RAW_DIR / "transactions.parquet"
if parquet.exists():
    raw = pd.read_parquet(parquet); source = "IEEE-CIS (real)"
else:
    from data.synthetic import generate_transactions
    raw = generate_transactions(n_users=3000, seed=42); source = "synthetic"
print("source:", source)

feats = build_features(raw)
tr, va, te = time_based_split(feats)
scaler = fit_scaler(tr)
train_ds, val_ds, test_ds = build_sequences(tr, scaler), build_sequences(va, scaler), build_sequences(te, scaler)

out = run_training(train_ds, val_ds, TrainConfig(max_epochs=12))
model = out.model.eval()

test_scores = _scores(model, test_ds, device="cpu")
threshold = tune_threshold(val_ds.y, _scores(model, val_ds, device="cpu"))  # FPR<=5% on val
y_pred = (test_scores >= threshold).astype(int)
res = evaluate(test_ds.y, test_scores, threshold=threshold)
print(f"test AUC={res.auc_roc:.4f}  threshold={threshold:.4f}  "
      f"precision={res.precision_at_threshold:.3f}  recall={res.recall_at_threshold:.3f}  fpr={res.fpr_at_threshold:.3f}")

## 2. Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_ds.y.astype(int), y_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for (r, c), v in np.ndenumerate(cm):
    ax.text(c, r, str(v), ha="center", va="center",
            color="white" if v > cm.max()/2 else "black", fontsize=13)
ax.set_xticks([0, 1]); ax.set_xticklabels(["pred legit", "pred fraud"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["true legit", "true fraud"])
ax.set_title("Confusion matrix @ deployed threshold"); plt.show()

## 3. Per-sequence context for error analysis

Build a tidy frame aligning each test user with their label, score, prediction, dollars at risk, and a few behavioral summaries.

In [ ]:
# dollars at risk per user = sum of fraudulent transaction amounts in their history
fraud_amt = (feats[feats[LABEL_COL] == 1]
             .groupby("user_id")[AMOUNT_COL].sum())
amount_by_user = pd.Series(fraud_amt, index=test_ds.user_ids).fillna(0.0).to_numpy()

# behavioral summaries from the (scaled) sequence, ignoring padding
def feat_mean(name):
    j = FEATURE_COLUMNS.index(name)
    X = test_ds.X[:, :, j].copy().astype(float)
    X[test_ds.mask] = np.nan
    return np.nanmean(X, axis=1)

df = pd.DataFrame({
    "user_id": test_ds.user_ids,
    "label": test_ds.y.astype(int),
    "score": test_scores,
    "pred": y_pred,
    "fraud_amount": amount_by_user,
    "seq_len": (~test_ds.mask).sum(axis=1),
    **{f"mean_{f}": feat_mean(f) for f in MONITORED_FEATURES},
})
df["bucket"] = np.select(
    [(df.label==1)&(df.pred==1), (df.label==1)&(df.pred==0),
     (df.label==0)&(df.pred==1), (df.label==0)&(df.pred==0)],
    ["TP", "FN", "FP", "TN"], default="?")
df["bucket"].value_counts()

## 4. False negatives — missed fraud

Compare missed fraud (FN) against caught fraud (TP): are misses lower-velocity, smaller-amount, or shorter-history cases?

In [ ]:
cols = ["score", "fraud_amount", "seq_len"] + [f"mean_{f}" for f in MONITORED_FEATURES]
summary = df[df.label==1].groupby("bucket")[cols].mean().T
print("Caught (TP) vs Missed (FN) — mean profile:")
display(summary[["TP", "FN"]] if "FN" in summary else summary)

fn = df[df.bucket=="FN"].sort_values("fraud_amount", ascending=False)
print(f"\n{len(fn)} false negatives; highest-$ misses:")
display(fn[["user_id","score","fraud_amount","seq_len"]].head(8))

## 5. False positives — legit flagged as fraud

Which benign sequences trip the model? Compare FP vs TN profiles.

In [ ]:
summary0 = df[df.label==0].groupby("bucket")[cols].mean().T
print("False alarm (FP) vs correct (TN) — mean profile:")
display(summary0[["FP", "TN"]] if "FP" in summary0 else summary0)

fp = df[df.bucket=="FP"].sort_values("score", ascending=False)
print(f"\n{len(fp)} false positives; most confident false alarms:")
display(fp[["user_id","score","seq_len"]+[f"mean_{f}" for f in MONITORED_FEATURES]].head(8))

## 6. Business metrics @ deployed threshold

In [ ]:
m = business_metrics(test_ds.y, y_pred, amount_by_user)
for k, v in m.as_dict().items():
    print(f"{k:>22}: {v:,.2f}" if isinstance(v, float) else f"{k:>22}: {v}")

# Optionally persist the artifact the plan logs to MLflow
# from training.business_metrics import save_business_metrics_json
# save_business_metrics_json(m, "business_metrics.json")

## 7. Profit curve — net savings vs threshold

The FPR≤5% threshold is the *deployment* rule; this shows the **profit-maximising** threshold, which may sit elsewhere depending on the false-positive cost.

In [ ]:
thr, savings = net_savings_by_threshold(test_ds.y, test_scores, amount_by_user)
best = thr[int(np.argmax(savings))]
plt.figure(figsize=(10, 4))
plt.plot(thr, savings, marker="o")
plt.axvline(threshold, color="green", ls="--", label=f"deployed (FPR≤5%)={threshold:.2f}")
plt.axvline(best, color="red", ls="--", label=f"max-profit={best:.2f}")
plt.xlabel("threshold"); plt.ylabel("net savings ($)"); plt.legend()
plt.title("Net savings vs threshold"); plt.show()
print(f"max-profit threshold: {best:.3f}  |  net savings there: ${savings.max():,.0f}")

## 8. Findings

> Fill in after running on real IEEE-CIS data.

- **False negatives** skew toward (low-velocity / small-amount / short-history)
  fraud — list the dominant missed pattern and whether it's worth a rule.
- **False positives** cluster around (new-device legit logins / travel spikes) —
  candidates for a allow-list or feature fix.
- **Net savings** at the deployed threshold = $____; the profit-maximising
  threshold (____) would change net savings by $____ but move FPR to ____%.
- **Decision:** keep the FPR≤5% threshold for customer-experience reasons, or
  shift toward the profit optimum? Document the call and the assumed FP cost.